# Introduction

Group members:
- Gianluca Carta
- Giorgio Coccapani
- Emma Fanfoni
- Riccardo Grossoni


The task: medical question answering.

The paper: https://arxiv.org/pdf/2304.08247.pdf

The provided paper states that LLMs can store knowledge useful for effectively answering questions in a specific domain (in the present case, medical) simply through fine tuning. What the paper states is very ambitious, because fine tuning is usually employed to teach the model how to behave, what output to provide to input presented in a given form. It does not “augment” the notional background of a model.

For this reason, to give the impression that the model has learned and assimilated a large quantity of notions in a certain domain, RAG (retrival augmented generation) is preferred. RAG makes it possible to produce more reliable answers supported by reliable sources, as well as to limit the phenomenon of hallucination, that is, outputs that contain incorrect information but are presented as if they were true. very dangerous expecially in the mdeical field. RAG uses user input to extract useful information to formulate an answer from a given authoritative source.

In the Notebook, we will use the techniques learned to expolate the data provided, and compare the performance of fine-tuned models with the data provided, and others where RAG is preferred.

NOTE🤨 : all the datasets and the finetuned model we cite later in the notebook is available here:
- [drive folder](https://drive.google.com/drive/folders/18YCZr3BC0N010S1VAViSUX1IX1ru0OXY?usp=sharing)

# Setup

Connecting to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
path = 'NLP/final'

os.chdir(f'/content/drive/MyDrive/{path}')
os.getcwd()

Mounted at /content/drive


'/content/drive/MyDrive/NLP/final'

Importing libraries

In [ ]:
!pip install nltk
!pip install datasets
!pip install transformers[torch]
!pip install tokenizers
!pip install evaluate
!pip install rouge_score
!pip install sentencepiece
!pip install huggingface_hub
!pip install cowsay
!pip install -q wordcloud
!pip install -q kneed
!pip install -q -U transformers bitsandbytes accelerate xformers
!pip install -q -U sentence-transformers
!pip install hnswlib

In [3]:
import nltk
import pprint
import evaluate
import pandas as pd
import numpy as np
import json
import copy
import string
import matplotlib.pyplot as plt
import pprint
import datasets
import gzip
import pickle
import torch
import random
import re
import ast
import math
import torch.nn.functional as F
import seaborn as sns
import hnswlib

from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter
from wordcloud import WordCloud
import cowsay
from sklearn import metrics
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.metrics import pairwise_distances_argmin_min
from kneed import KneeLocator

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Loading files

In [4]:
# Paths

step_1_qa = 'docs/steps/step1.json'
step_1_sol = 'docs/steps/step1_solutions.json'
step_2_qa = 'docs/steps/step2.json'
step_2_sol = 'docs/steps/step2_solutions.json'
step_3_qa = 'docs/steps/step3.json'
step_3_sol = 'docs/steps/step3_solutions.json'

wikidoc_path = 'docs/flashcards/wikidoc_dataset.json'
original_path = 'docs/flashcards/original.json'
medqa_path = 'docs/flashcards/medqa_dataset.json'
medqa_cleaned_path = 'docs/flashcards/medqa_dataset_cleaned.json'
books_path = 'docs/books'

with open(step_1_qa, "r") as file_data:
    step1 = json.load(file_data)
with open(step_2_qa, "r") as file_data:
    step2 = json.load(file_data)
with open(step_3_qa, "r") as file_data:
    step3 = json.load(file_data)

with open(step_1_sol, "r") as file_solutions:
    step1_sol = json.load(file_solutions)
with open(step_2_sol, "r") as file_solutions:
    step2_sol = json.load(file_solutions)
with open(step_3_sol, "r") as file_solutions:
    step3_sol = json.load(file_solutions)


with open(wikidoc_path, "r") as file_data:
    wikidoc = json.load(file_data)
with open(original_path, "r") as file_data:
    original = json.load(file_data)
with open(medqa_path, "r") as file_data:
    medqa = json.load(file_data)
with open(medqa_cleaned_path, "r") as file_data:
    medqa_cleaned = json.load(file_data)


**Questions cleaning**

From a manual analysis of the dataset that will be used for testing, we noticed that some questions are not suitable for testing, for a variety of reasons.

We therefore created and applied functions that would remove:
- questions referring to images
- questions that have missing answers

In [ ]:
def remove_faulty(data, sol_dataset):
  # List to store the indices of removed objects
  removed_indices = []
  # Iterate through the original dataset to find indices of removed objects
  for index, item in enumerate(data):
    if 'image_url' in item or 'image' in item or not item.get('options') or ('options' in item and not any(item['options'].values())):
        removed_indices.append(index)

  # Remove solutions corresponding to removed questions
  for index in sorted(removed_indices, reverse=True):
      question_number = str(index + 1)  # Adjust index to question number
      if question_number in sol_dataset:
        del sol_dataset[question_number]
  new_sol_dataset = {}
  for i, (key, value) in enumerate(sol_dataset.items(), start=1):
      new_sol_dataset[str(i)] = value

  data[:] = [item for item in data if ('image_url' not in item and 'image' not in item) and ('options' in item and any(item['options'].values()))]
  return new_sol_dataset

def clean_dataset(data):
  cleaned_data = [item for item in data if item.get('input') != "" and item.get('output') != ""]
  data[:]=cleaned_data


step1_sol=remove_faulty(step1, step1_sol)
step2_sol=remove_faulty(step2, step2_sol)
step3_sol=remove_faulty(step3, step3_sol)
clean_dataset(original)